In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

# Load and prepare data (same as modeling notebook)
df = pd.read_csv('../data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.drop('customerID', axis=1, inplace=True)

le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Re-train best model (XGBoost)
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

In [ ]:
# SHAP Explainer
explainer = shap.TreeExplainer(xgb_model)

# Note: For XGBoost, shap_values is often returned as a single 2D array (N, M)
# for the probability of class 1. 
shap_values = explainer.shap_values(X_test)

print(f"X_test shape: {X_test.shape}")
if isinstance(shap_values, list):
    print(f"shap_values is a list of {len(shap_values)} arrays. Class 1 shape: {shap_values[1].shape}")
    plot_values = shap_values[1]
else:
    print(f"shap_values shape: {shap_values.shape}")
    plot_values = shap_values

In [ ]:
# Global Feature Importance (Bar Plot)
plt.figure(figsize=(10, 6))
shap.summary_plot(
    plot_values, 
    X_test, 
    plot_type="bar", 
    show=False
)
plt.title("Top Churn Drivers — Global Feature Importance")
plt.tight_layout()
plt.savefig('../outputs/shap_summary.png', dpi=150)
plt.show()

In [ ]:
# SHAP Value Distribution (Dot Plot)
plt.figure()
shap.summary_plot(
    plot_values, # Using plot_values to ensure shape consistency
    X_test,
    show=False
)
plt.title("SHAP Value Distribution by Feature")
plt.tight_layout()
plt.savefig('../outputs/shap_summary_dot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Individual Prediction Explanation (Force Plot)
customer_index = 0

# Determine expected value and shap values for class 1
if isinstance(explainer.expected_value, (list, np.ndarray)) and len(explainer.expected_value) > 1:
    ev = explainer.expected_value[1]
else:
    ev = explainer.expected_value

shap.initjs()
force_plot = shap.force_plot(
    ev,
    plot_values[customer_index],
    X_test.iloc[customer_index],
    matplotlib=True,
    show=False
)
plt.savefig('../outputs/shap_single_customer.png', dpi=150, bbox_inches='tight')
plt.show()

In [13]:
# This feeds directly into the LangChain agent later
feature_importance = pd.DataFrame({
    'feature': X_test.columns,
    'mean_shap': np.abs(plot_values).mean(axis=0)
}).sort_values('mean_shap', ascending=False)

print("Top 5 Churn Drivers:")
print(feature_importance.head(5).to_string(index=False))

# Save for LangChain agent
feature_importance.to_csv('../outputs/shap_feature_importance.csv', index=False)

Top 5 Churn Drivers:
       feature  mean_shap
      Contract   1.068554
        tenure   0.600774
MonthlyCharges   0.533560
  TotalCharges   0.417525
OnlineSecurity   0.304276
